## ライブラリインポート

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

## 環境準備

In [11]:
# .envファイルを読み込む
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

# クライアントの準備
client = OpenAI(api_key=openai_api_key)

## テキスト生成

In [ ]:
# メッセージリストの準備
messages = [
    {
        "role": "developer",
        "content": "優秀なAIアシスタントです。"
    },
    {
        "role": "user",
        "content": "日本一高い山は？",
    }
]
# 推論の実行
response = client.responses.create(
    model="gpt-4o",
    input=messages
)
print(response.output_text)

日本一高い山は富士山です。高さは3,776メートルです。


## 画像認識

In [10]:
# メッセージリストの準備
messages = [
    {
        "role": "user", 
        "content": "これは何の画像ですか？"
    },
    {
        "role": "user",
        "content": [
            {
                "type": "input_image",
                "image_url": "https://assets.st-note.com/img/1699330153755-AtaoOsjEq4.jpg"
            }
        ]
    }
]

# 推論の実行
response = client.responses.create(
    model="gpt-4o",
    input=messages
)
print(response.output_text)

この画像には、テレビの前で横たわっている猫が写っています。猫の横には、ゲーム機の一部（Nintendo Switchのコントローラーとドック）が見えます。猫は、白と黒の模様が特徴的です。


## Tool

In [12]:
# メッセージリストの準備
messages = [
    {
        "role": "user",
        "content": "本日のAI関連のニュースは？",
    }
]

# 推論の実行
response = client.responses.create(
    model="gpt-4o",
    tools=[{"type": "web_search_preview"}],
    input=messages
)
print(response.output_text)

本日（2025年3月15日）時点でのAI関連の最新ニュースは以下のとおりです。

**1. AppleのAI時代が到来：新機能のプレビューとChatGPT統合で未来を切り開く**

Appleは、独自の生成AI「Apple Intelligence」を発表し、ChatGPTとの連携を開始しました。これにより、iOSデバイス上での音声アシスタント「Siri」やテキスト・画像生成機能が強化され、ユーザー体験が向上しています。 ([nipponai.jp](https://nipponai.jp/special/ai%E3%83%8B%E3%83%A5%E3%83%BC%E3%82%B9/?utm_source=openai))

**2. シーメンスとマイクロソフトの提携で、産業AIが変革の時代へ**

シーメンスとマイクロソフトが提携し、製造業におけるAIの革新を推進しています。この協力により、産業分野での自動化とスマート化が加速し、労働力不足の解消や生産性の向上が期待されています。 ([nipponai.jp](https://nipponai.jp/special/ai%E3%83%8B%E3%83%A5%E3%83%BC%E3%82%B9/?utm_source=openai))

**3. グーグルが核エネルギーに注目：AIデータセンターの未来のエネルギーソリューション**

グーグルは、AIデータセンターの膨大な電力需要を満たすため、クリーンで安定したエネルギー源として核エネルギーの活用を検討しています。これにより、持続可能なエネルギー供給と環境負荷の低減が期待されています。 ([nipponai.jp](https://nipponai.jp/special/ai%E3%83%8B%E3%83%A5%E3%83%BC%E3%82%B9/?utm_source=openai))

**4. AIが未来をリード：日本の電子＆モビリティ展示会が初の共同開催、技術と自動車産業の深い融合**

日本で開催された電子＆モビリティ展示会（CEATEC）では、AI技術と自動車産業の融合が強調されました。これにより、次世代のモビリティソリューションやスマートシティの実現に向けた取り組みが加速しています。 ([nipponai.jp](https://nipponai.jp/

## ストリーミング

In [13]:
# メッセージリストの準備
messages = [
    {
        "role": "user",
        "content": "早口言葉を言ってください。",
    }
]

# 推論の実行
stream = client.responses.create(
    model="gpt-4o",
    input=messages,
    stream=True,
)
for event in stream:
   if event.type == 'response.output_text.delta':
        print(event.delta, end='')

もちろん！有名な早口言葉の一つを紹介しますね。

「生麦、生米、生卵」

ほかにも、

「バスガス爆発」

や

「赤巻紙、青巻紙、黄巻紙」

こんなのもあります。試してみてください！

## エージェント

In [15]:
# asyncio.run()を使うための前準備
import nest_asyncio
nest_asyncio.apply()

In [17]:
from agents import Agent, Runner
import asyncio

# 日本語エージェント
japanese_agent = Agent(
    name="日本語エージェント",
    instructions="あなたは日本語しか話せません。",
)

# 英語エージェント
english_agent = Agent(
    name="英語エージェント",
    instructions="あなたは英語しか話せません。",
)

# トリアージエージェント
triage_agent = Agent(
    name="トリアージエージェント",
    instructions="リクエストの言語に基づいて適切なエージェントに引き継ぎます。",
    handoffs=[japanese_agent, english_agent],
)

In [19]:
# メイン
async def main():
    result = await Runner.run(triage_agent, input="今日はsuper exciting!!")
    print(result.final_output)

# メインの実行
asyncio.run(main())

That's great to hear! What's happening today that's super exciting?


## 会話

In [23]:
# 推論の実行
response = client.responses.create(
    model="gpt-4o-mini",
    input=[{"role": "user", "content": "日本一高い山は？"}],
)
print(response.output_text)

日本一高い山は富士山です。標高は3,776メートルで、静岡県と山梨県にまたがっています。富士山はその美しい姿から、古くから信仰の対象となり、多くの人々に愛されています。


In [26]:
# 推論の実行
second_response = client.responses.create(
    model="gpt-4o",
    previous_response_id=response.id,
    input=[{"role": "user", "content": "二番目に高い山は？"}],
)
print(second_response.output_text)

日本で二番目に高い山は北岳です。標高は3,193メートルで、山梨県の南アルプスに位置しています。北岳は、登山愛好家に人気があり、自然の美しさを楽しむことができる場所です。


In [27]:
# 推論の実行
third_response = client.responses.create(
    model="gpt-4o",
    previous_response_id=second_response.id,
    input=[{"role": "user", "content": "一番長い川は？"}],
)
print(third_response.output_text)

日本で一番長い川は信濃川です。長さは367キロメートルで、新潟県と長野県を流れています。新潟県では、「信濃川」と呼ばれ、長野県では「千曲川」として知られています。


## Function Calling

In [36]:
import requests

# 関数の定義
def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [37]:
# ツールリストの準備
tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "指定された座標の現在の温度を摂氏で取得します。",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

# メッセージリストの準備
messages = [
    {"role": "user", "content": "今日の大阪の天気は？"}
]

In [38]:
# 推論の実行
response = client.responses.create(
    model="gpt-4o",
    input=messages,
    tools=tools
)
print(response)

Response(id='resp_67d5746ace388191b0b49fb250efdf3f01b691d4701e00cc', created_at=1742042218.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-2024-08-06', object='response', output=[ResponseFunctionToolCall(id='fc_67d5746b52548191b5e04d3ec7bbc88e01b691d4701e00cc', arguments='{"latitude":34.6937,"longitude":135.5023}', call_id='call_HFLSGvXSE7iobmVCmcocpbTD', name='get_weather', type='function_call', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'latitude': {'type': 'number'}, 'longitude': {'type': 'number'}}, 'required': ['latitude', 'longitude'], 'additionalProperties': False}, strict=True, type='function', description='指定された座標の現在の温度を摂氏で取得します。')], top_p=1.0, max_output_tokens=None, previous_response_id=None, reasoning=Reasoning(effort=None, generate_summary=None), status='completed', text=ResponseTextConfig(format=ResponseFormat

In [39]:
import json

# 関数の実行
tool_call = response.output[0]
args = json.loads(tool_call.arguments)
result = get_weather(args["latitude"], args["longitude"])
print(result)

5.9


In [40]:
# メッセージリストに関数の実行結果を追加
messages.append(tool_call)  # Function Callingの応答
messages.append({           # 関数の実行結果を追加
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

# 推論の実行
response_2 = client.responses.create(
    model="gpt-4o",
    input=messages,
    tools=tools,
)
print(response_2.output_text)

今日の大阪の現在の気温は約5.9°Cです。天気の詳細については、具体的な天気予報を確認することをお勧めします。
